# FinQA: test the distilled model again (no training)

Use this only if the testing step of `finqa_distilled.ipynb` stopped. It loads the adapter you already downloaded and answers the 297 test questions three times each, 8 at a time.

1. **Runtime > Change runtime type > T4 GPU**, save.
2. **Runtime > Run all**. When the upload box appears, pick `fin_distilled_adapter.zip`.
3. Keep the tab open. `fin_distilled_results.zip` downloads at the end: send it to Claude.

Every log line starts with the time (India) and shows done / total.

In [ ]:
%%capture
!pip install -q unsloth
!rm -rf /content/slm-opti && git clone -q --depth 1 -b v2-split https://github.com/eshwarpotturi/slm-opti /content/slm-opti
!rm -rf /content/FinQA && git clone -q --depth 1 https://github.com/czyssrs/FinQA /content/FinQA

In [ ]:
import os, sys, json, time, re, random, zipfile, collections
os.environ["TZ"] = "Asia/Kolkata"; time.tzset()
os.chdir("/content/slm-opti"); sys.path.insert(0, "/content/slm-opti"); sys.path.insert(0, "/content/slm-opti/fin")
def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)
import torch
assert torch.cuda.is_available(), "No GPU. Choose Runtime > Change runtime type > T4 GPU, then Runtime > Run all."
from unsloth import FastModel
model, tokenizer = FastModel.from_pretrained(model_name="unsloth/gemma-3-4b-it", max_seq_length=3072, load_in_4bit=True, full_finetuning=False)
import run, prep
log("model loaded")

In [ ]:
from google.colab import files
log("pick fin_distilled_adapter.zip in the upload box")
up = files.upload()
os.makedirs("/content/fin_distilled_adapter", exist_ok=True)
zipfile.ZipFile(next(iter(up)) if not next(iter(up)).startswith("/") else next(iter(up))).extractall("/content/fin_distilled_adapter")
from peft import PeftModel
model = PeftModel.from_pretrained(model, "/content/fin_distilled_adapter")
log("adapter loaded")

## The 297 unseen questions: three answers each, 8 questions at a time

In [ ]:
FastModel.for_inference(model)
tok = getattr(tokenizer, "tokenizer", tokenizer)
tok.padding_side = "left"
tests = json.load(open("fin/data/test.json"))
OUT = "fin/results/gemma-3-4b-distilled__L24-dist.jsonl"
done = {json.loads(l)["id"] for l in open(OUT)} if os.path.exists(OUT) else set()
todo = sorted([t for t in tests if t["id"] not in done], key=lambda t: len(run.render(t)))
BATCH = 8

def prompt(t):
    msgs = [{"role": "user", "content": [{"type": "text", "text": run.TUNED_DIST + "\n\n" + run.ask(t)}]}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def generate(texts, sample, n):
    enc = tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
    kw = dict(max_new_tokens=350, num_return_sequences=n, pad_token_id=tok.pad_token_id)
    kw.update(do_sample=True, temperature=0.7) if sample else kw.update(do_sample=False)
    out = model.generate(**enc, **kw)
    return tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)

def read(text):
    yn = re.findall(r"ANSWER\s*:\s*\**\s*(yes|no)\b", text or "", flags=re.I)
    if yn:
        return yn[-1].lower(), None
    f = run.read_formula(text)
    if not f:
        return None, None
    f = f.split("=")[0].strip()
    try:
        v = run.calc(f)
        return (None if isinstance(v, str) else v), f
    except Exception:
        return None, f

def key(a):
    return "%.3g" % a if isinstance(a, float) else a

log("testing: %d to do, %d already done" % (len(todo), len(done)))
t0 = time.time()
for i in range(0, len(todo), BATCH):
    part = todo[i:i + BATCH]
    texts = [prompt(t) for t in part]
    t1 = time.time()
    firsts = generate(texts, False, 1)
    others = generate(texts, True, 2)
    for j, t in enumerate(part):
        replies = [firsts[j], others[2 * j], others[2 * j + 1]]
        tries = [read(r) for r in replies]
        counts = collections.Counter(key(a) for a, _ in tries if a is not None)
        if counts:
            top = counts.most_common(1)[0][0]
            a, f = next(x for x in tries if x[0] is not None and key(x[0]) == top)
        else:
            a, f = None, None
        rec = {"id": t["id"], "gold": t["answer"], "answer": a, "formula": f, "right": run.correct(a, t["answer"]),
               "first": tries[0][0], "first_formula": tries[0][1], "first_right": run.correct(tries[0][0], t["answer"]),
               "agree": counts[top] if counts else 0, "votes": list(counts.items()), "raw": replies[0][-800:], "cost": 0.0,
               "secs": round((time.time() - t1) / len(part), 1)}
        open(OUT, "a").write(json.dumps(rec) + "\n")
    rs = [json.loads(l) for l in open(OUT)]
    log("%d/%d done  first answer right %d (%.0f%%)  batch %.0fs" % (len(rs), len(tests), sum(r["first_right"] for r in rs),
        100 * sum(r["first_right"] for r in rs) / len(rs), time.time() - t1))
log("testing took %.0f minutes" % ((time.time() - t0) / 60))

In [ ]:
rs = [json.loads(l) for l in open(OUT)]
first = sum(r["first_right"] for r in rs)
sure = [r for r in rs if r["agree"] == 3]
log("first answer correct: %d of %d (%.0f%%)" % (first, len(rs), 100 * first / len(rs)))
log("all three answers agree on %d questions; %d of those are correct" % (len(sure), sum(r["right"] for r in sure)))
with zipfile.ZipFile("/content/fin_distilled_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in [OUT]:
        z.write(f)
from google.colab import files
files.download("/content/fin_distilled_results.zip")